<a href="https://colab.research.google.com/github/Nour-Tamimi/BinXtraining/blob/main/Week/Day2/Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")
print(dataset)
print(dataset["train"][0])
print(dataset["train"].features)  # shows the label names (World, Sports, Business, Sci/Tech)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


In [11]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# Standard English stopword list...
stop_words = set(stopwords.words('english'))

# ...but carve out negation words so they survive cleaning
negation_words = {
    "no", "not", "nor", "none", "never", "n't",
    "don", "don't", "doesn", "doesn't", "didn", "didn't",
    "isn", "isn't", "aren", "aren't", "wasn", "wasn't", "weren", "weren't",
    "won", "won't", "wouldn", "wouldn't", "can", "can't", "cannot",
    "couldn", "couldn't", "shouldn", "shouldn't", "hasn", "hasn't",
    "haven", "haven't", "hadn", "hadn't", "mustn", "mustn't"
}
stop_words = stop_words - negation_words

lemmatizer = WordNetLemmatizer()

def clean_pipeline(text):
    text = text.lower()
    text = re.sub(r"#\d+;", " ", text)          # kill leftover HTML entities like #39;
    text = re.sub(r"[^a-z0-9\s']", " ", text)    # strip all punctuation except apostrophe (keep n't)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

# quick test
sample = dataset["train"][0]["text"]
print(sample)
print(clean_pipeline(sample))

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
['wall', 'st', 'bear', 'claw', 'back', 'black', 'reuters', 'reuters', 'short', 'seller', 'wall', 'street', "'s", 'dwindling', 'band', 'ultra', 'cynic', 'seeing', 'green']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
def clean_batch(batch):
    return {'cleaned_text': [clean_pipeline(t) for t in batch['text']]}


cleaned_train_dataset = dataset['train'].map(clean_batch, batched=True, batch_size=1000)
cleaned_test_dataset = dataset['test'].map(clean_batch, batched=True, batch_size=1000)

print(cleaned_train_dataset[0]["cleaned_text"])

['wall', 'st', 'bear', 'claw', 'back', 'black', 'reuters', 'reuters', 'short', 'seller', 'wall', 'street', "'s", 'dwindling', 'band', 'ultra', 'cynic', 'seeing', 'green']


In [13]:
#Step 1
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import time

# Use your cleaned text from the AG News pipeline
train_texts = cleaned_train_dataset["cleaned_text"]
train_labels = cleaned_train_dataset["label"]

test_texts = cleaned_test_dataset["cleaned_text"]
test_labels = cleaned_test_dataset["label"]

# To keep this comparable to your LSTM (trained on 5000 examples),
# subsample train set the same way
import random
random.seed(42)
idx = random.sample(range(len(train_texts)), 5000)

# Join the lists of tokens back into strings
train_texts_sub = [" ".join(train_texts[i]) for i in idx]
train_labels_sub = [train_labels[i] for i in idx]

# Join the lists of tokens back into strings for the test set
test_texts_processed = [" ".join(doc) for doc in test_texts]

# Vectorize
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_texts_sub)
X_test = vectorizer.transform(test_texts_processed)

# Train
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_labels_sub)

# Evaluate
preds = clf.predict(X_test)
acc = accuracy_score(test_labels, preds)

print(f"TF-IDF + Logistic Regression")
print(f"Accuracy: {acc*100:.2f}%")
print(classification_report(test_labels, preds))

TF-IDF + Logistic Regression
Accuracy: 87.76%
Training time: 1.0 seconds
              precision    recall  f1-score   support

           0       0.89      0.88      0.88      1900
           1       0.91      0.97      0.94      1900
           2       0.85      0.83      0.84      1900
           3       0.85      0.84      0.85      1900

    accuracy                           0.88      7600
   macro avg       0.88      0.88      0.88      7600
weighted avg       0.88      0.88      0.88      7600



## TF-IDF + Logistic Regression Results

**Accuracy: 87.76%** — out of 7,600 test articles, the model correctly
predicted the topic for about 88 out of every 100.

### Reading the table

- **precision**: of all the articles the model *labeled* as this class,
  how many actually were that class? High precision = few false alarms.
- **recall**: of all the articles that *actually are* this class, how
  many did the model catch? High recall = few missed cases.
- **f1-score**: a single number that balances precision and recall
  (their harmonic mean) — useful when you want one score per class.
- **support**: how many real test examples exist for that class
  (1,900 each here, since AG News is perfectly balanced across its
  4 topics).

### Per-class breakdown

| Class | Topic     | F1-score | Notes |
|-------|-----------|----------|-------|
| 0     | World     | 0.88     | Strong |
| 1     | Sports    | 0.94     | Best — sports vocabulary is distinctive |
| 2     | Business  | 0.84     | Weakest — overlaps with Sci/Tech |
| 3     | Sci/Tech  | 0.85     | Overlaps with Business (companies, products) |

**Why Sports scores highest:** sports articles use very distinctive
words (team names, "goal," "match," "win") that rarely appear in other
categories — TF-IDF picks up on this easily.

**Why Business and Sci/Tech are harder:** these two categories share a
lot of vocabulary — company names, "launch," "market," "shares" — so
even a strong baseline struggles to fully separate them.

### accuracy vs. macro avg vs. weighted avg

Since every class has exactly 1,900 test examples, the dataset is
**perfectly balanced** — so `macro avg` (average of the 4 classes,
equal weight) and `weighted avg` (average weighted by class size) come
out identical here. That won't always be true on imbalanced datasets.

In [15]:
!pip install gensim
import gensim.downloader as api

# This downloads ~66MB (glove-wiki-gigaword-100) the first time, then caches it
glove_vectors = api.load("glove-wiki-gigaword-100")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 66.0 MB/s eta 0:00:00
[==================================================] 100.0% 128.1/128.1MB downloaded


In [16]:
# Pick a few words relevant to AG News topics (World, Sports, Business, Sci/Tech)
words_to_check = ["king", "computer", "stock", "team", "war"]

for word in words_to_check:
    print(f"\nNearest neighbors of '{word}':")
    similar = glove_vectors.most_similar(word, topn=5)
    for neighbor, score in similar:
        print(f"  {neighbor}: {score:.3f}")


Nearest neighbors of 'king':
  prince: 0.768
  queen: 0.751
  son: 0.702
  brother: 0.699
  monarch: 0.698

Nearest neighbors of 'computer':
  computers: 0.875
  software: 0.837
  technology: 0.764
  pc: 0.737
  hardware: 0.729

Nearest neighbors of 'stock':
  shares: 0.853
  stocks: 0.831
  market: 0.799
  exchange: 0.785
  trading: 0.763

Nearest neighbors of 'team':
  teams: 0.852
  squad: 0.785
  football: 0.772
  players: 0.766
  coach: 0.765

Nearest neighbors of 'war':
  wars: 0.769
  conflict: 0.766
  invasion: 0.743
  military: 0.737
  occupation: 0.730


these words are related in meaning, not in spelling or frequency. TF-IDF would treat "stock" and "shares" as two completely unrelated columns in a matrix — it has no idea they're connected. GloVe places them close together because they tend to appear in similar surrounding words across huge amounts of text.

## Step 3: Model Comparison

| Model | Accuracy | Training Required | Time |
|---|---|---|---|
| Transformer (zero-shot, bart-large-mnli) | 70.50% | None | 30 sec |
| LSTM (trained from scratch, 5000 examples) | 80.00% | Yes (5 epochs) | ~1 min |
| TF-IDF + Logistic Regression (5000 examples) | 87.76% | Yes (1 pass, no epochs) | 1.0 sec |
| DistilBERT (fine-tuned, 5000 examples) | 90.83% | Yes (3 epochs) | few min (GPU) |

**Key takeaway:** accuracy alone doesn't tell the whole story — the right
choice depends on the constraint that matters most:

- **Need it instantly, no labeled data at all?** → Zero-shot transformer.
  Weakest accuracy, but the only option needing zero training examples.
- **Need it fast and cheap, with some labeled data?** → TF-IDF. Nearly as
  accurate as the fine-tuned transformer, but trains in ~1 second instead
  of minutes — a genuinely strong, practical baseline.
- **Need the best possible accuracy, and have GPU + time to spare?** →
  Fine-tuned transformer. Best result here, because it combines pre-trained
  language knowledge with task-specific training.
- **LSTM's role:** useful as a "trained from scratch" reference point — it
  shows what's achievable with sequence modeling but no pre-training,
  which turned out to be the weakest of the three trained models.

## Step 4: Which Representation Fits This Project?

For a **text classification task like AG News** (4 balanced topic classes,
5000 labeled training examples), **TF-IDF + Logistic Regression is the
recommended first choice**, with the fine-tuned transformer as a strong
upgrade if resources allow.

**Reasoning:**
- TF-IDF reached 87.76% accuracy in **1 second** of training — a remarkable
  efficiency-to-performance ratio, and more than good enough for most
  practical use cases.
- The fine-tuned transformer only improved on that by ~3 points (90.83%),
  at the cost of needing a GPU and several minutes of training — a
  reasonable trade if that extra accuracy matters for the use case
  (e.g., a production system), but often unnecessary for a baseline or
  prototype.
- The LSTM underperformed both simpler and more sophisticated approaches,
  showing that training a sequence model *from scratch* on only 5000
  examples isn't enough data for it to learn useful patterns on its own —
  it has no pre-trained knowledge to fall back on.
- The zero-shot transformer is the clear choice **only when no labeled
  data exists at all** — otherwise, any model that gets to train on your
  actual data outperforms it.

**Bottom line:** representation choice depends on your constraints, not
just raw accuracy. Here, TF-IDF is the pragmatic winner; fine-tuned
transformers are the accuracy ceiling when you can afford the extra cost.